In [1]:
import pandas as pd

data = pd.read_csv("gold_data.csv", index_col=0 , parse_dates=True)

print(data.info())
print(data.head())


<class 'pandas.core.frame.DataFrame'>
Index: 13757 entries, Ticker to 2025-11-03 19:00:00+00:00
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Close   13756 non-null  object
 1   High    13756 non-null  object
 2   Low     13756 non-null  object
 3   Open    13756 non-null  object
 4   Volume  13756 non-null  object
dtypes: object(5)
memory usage: 644.9+ KB
None
                                        Close               High  \
Price                                                              
Ticker                                   GC=F               GC=F   
Datetime                                  NaN                NaN   
2023-06-14 04:00:00+00:00  1961.9000244140625  1962.699951171875   
2023-06-14 05:00:00+00:00   1959.300048828125  1962.300048828125   
2023-06-14 06:00:00+00:00              1959.0  1960.800048828125   

                                          Low                Open Volume  
Price                

C:\Users\kioko\AppData\Local\Temp\ipykernel_12960\3957178863.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data = pd.read_csv("gold_data.csv", index_col=0 , parse_dates=True)


In [1]:
import yfinance as yf

def load_data(path=DATA_CSV):
    if os.path.exists(path):
        print(f"Loading data from {path}")
        df = pd.read_csv(path, index_col=0, parse_dates=True)
        # Normalize column names (common yfinance format sometimes capitalized)
        df.columns = [c.lower() for c in df.columns]
        # expect columns: open, high, low, close, volume (lowercase)
        if 'close' not in df.columns and ('adj close' in df.columns):
            df['close'] = df['adj close']
    else:
        print("Local CSV not found — trying yfinance fallback (GLD, last 730d hourly)")
        df = yf.download("GLD", period="730d", interval="1h")
        df.columns = [c.lower() + ("" if isinstance(c, str) else "") for c in df.columns]
    # Standardize to expected names
    rename_map = {}
    for col in df.columns:
        if col.lower().startswith('open'):
            rename_map[col] = 'open'
        if col.lower().startswith('high'):
            rename_map[col] = 'high'
        if col.lower().startswith('low'):
            rename_map[col] = 'low'
        if col.lower().startswith('close') or col.lower().startswith('adj close'):
            rename_map[col] = 'close'
        if col.lower().startswith('volume'):
            rename_map[col] = 'volume'
    df = df.rename(columns=rename_map)
    df = df[['open','high','low','close'] + ([ 'volume'] if 'volume' in df.columns else [])]
    df = df.sort_index()
    # Ensure hourly frequency index (fill small gaps)
    df = df[~df.index.duplicated(keep='first')]
    df = df.asfreq('1H')
    df = df.fillna(method='ffill')
    return df


_df = load_data()
print('Loaded rows:', len(_df))
print(_df.head())

NameError: name 'DATA_CSV' is not defined

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from datetime import timedelta

# indicators
import ta

# modeling
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_fscore_support
from sklearn.calibration import CalibratedClassifierCV

ModuleNotFoundError: No module named 'matplotlib'